[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/02_synthetic_qa_generation.ipynb)

# Step 2 — Synthetic Q&A Generation Strategies

Compare how a **teacher LLM** generates policy Q&A using different prompting strategies, and save the raw outputs as **training** data for the distillation pipeline (filtered in Step 3, used to fine-tune the small model later).

## Scope: training data only
This notebook operates exclusively on **TRAIN-split paragraphs**. It is *not* where the evaluation/test set is built — that happens in Step 1 via a separate `generate_test_qa_batch` path that targets specific small-model `FailureMode`s (format non-compliance, domain vocabulary drift, refusal calibration, multi-constraint collapse).

Because the `QASample` schema is shared across train and test samples (one JSONL shape, one set of loaders/filters), training samples still carry a `failure_mode` field — but it will be `None` here, since failure-mode targeting is a test-set concern. Don't be surprised when cell 8 prints `Failure mode: None` for every strategy.

## Learning objectives
- Zero-shot, one-shot, few-shot, and topic-controlled generation
- Understand distillation: strong model → synthetic training data → small model
- See why train-side `failure_mode` is `None` while test-side samples set it explicitly

In [1]:
import os
from pathlib import Path

from aieng.syn_data.text import (
    load_implementation_dotenv,
    PARAGRAPHS_PATH,
    SYNTHETIC_RAW_PATH,
    Paragraph,
    ParagraphSplit,
    compare_generation_strategies,
    create_teacher_client,
    generate_raw_synthetic_corpus,
    load_typed_jsonl,
    save_typed_jsonl,
    use_repo_root,
)
from rich.console import Console
from rich.panel import Panel
from rich.table import Table


# Setting the notebook directory to the project's root folder
if Path("").absolute().name == "synthetic-data-bootcamp":
    print(f"Notebook path is already the root path: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"The notebook path has been set to: {Path('').absolute()}")

load_implementation_dotenv()
use_repo_root(Path("."))

console = Console(width=100)

The notebook path has been set to: /home/coder/synthetic-data-bootcamp


In [ ]:
# TODO: remove this before merging into main

%load_ext autoreload
%autoreload 2

In [2]:
import logging


logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,  # override any earlier basicConfig from other libs
)

## 1. Load TRAIN paragraphs only

Use paragraphs saved in Step 1 where `split == train`.

In [3]:
all_paragraphs = load_typed_jsonl(PARAGRAPHS_PATH, Paragraph.from_dict)
train_paragraphs = [p for p in all_paragraphs if p.split == ParagraphSplit.TRAIN]
console.print(f"Train paragraphs available: {len(train_paragraphs)}")

para = train_paragraphs[0]

table = Table(title="Train Paragraph [0]", show_lines=True)
table.add_column("Field", style="bold cyan")
table.add_column("Value", style="white")

table.add_row("doc_id", str(para.doc_id))
table.add_row("para_id", str(para.para_id))
table.add_row("role", str(para.role))
table.add_row("split", str(para.split))
table.add_row("index", str(para.index))
table.add_row("text", para.text if len(para.text) < 500 else para.text[:500] + " ...")

console = Console()
console.print(Panel(table, title="First Train Paragraph Overview"))

Train paragraphs available: 36

╭──────────────────────────────────────── First Train Paragraph Overview ─────────────────────────────────────────╮
│                                               Train Paragraph [0]                                               │
│ ┏━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ │
│ ┃ Field   ┃ Value                                                                                             ┃ │
│ ┡━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ │
│ │ doc_id  │ cfpb_credit_card_agreement                                                                        │ │
│ ├─────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤ │
│ │ para_id │ cfpb_credit_card_agreement::p0000                                                                 │ │
│ ├─────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤ │
│ │ role    │ policy_dense                                                                                      │ │
│ ├─────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤ │
│ │ split   │ train                                                                                             │ │
│ ├─────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤ │
│ │ index   │ 0                                                                                                 │ │
│ ├─────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤ │
│ │ text    │ VISA CARD                                                                                         │ │
│ │         │ CONSUMER CREDIT CARD AGREEMENT                                                                    │ │
│ │         │ In this Agreement, “Agreement” means this Consumer Credit Card Agreement. “Disclosure” means the  │ │
│ │         │ Credit Card                                                                                       │ │
│ │         │ Account Opening Disclosure. The Account Opening Disclosure is incorporated into this Consumer     │ │
│ │         │ Credit Card Agreement                                                                             │ │
│ │         │ and is part of the Agreement. In this Agreement the words "you," “your,” and "yours" mean each    │ │
│ │         │ and all of those who                                                                              │ │
│ │         │ agree to be bound by this Agreement; "card" means the Visa credit card and any duplicates,        │ │
│ │         │ renewals, or substitutions  ...                                                                   │ │
│ └─────────┴───────────────────────────────────────────────────────────────────────────────────────────────────┘ │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 2. Compare generation strategies on the same paragraph

Each strategy is called **without** a `failure_mode`, so the printed `Failure mode: None` is expected — that field is only populated by the test-set construction path in Step 1. Here, what matters is the question/answer shape each strategy produces from the same source paragraph.

In [4]:
teacher = create_teacher_client()
demo_paragraph = train_paragraphs[0]

strategy_outputs = compare_generation_strategies(teacher, demo_paragraph)
for strategy, sample in strategy_outputs.items():
    from rich.text import Text

    header = Text(f"{strategy}", style="bold green")
    question = Text(f"Q: {sample.question}", style="cyan")
    answer = Text(f"A: {sample.gold_answer}", style="magenta")
    failure_mode = Text(f"Failure mode: {sample.failure_mode}", style="yellow")

    console.print(
        Panel(
            Text.assemble(header, "\n", question, "\n", answer, "\n", failure_mode),
            title=f"Generation Strategy: {strategy}",
            border_style="green",
        )
    )

2026-06-25 21:49:54,094 DEBUG urllib3.connectionpool: Starting new HTTPS connection (1): proxy.vectorinstitute.ai:443
2026-06-25 21:49:55,743 DEBUG urllib3.connectionpool: https://proxy.vectorinstitute.ai:443 "POST /v1/chat/completions HTTP/1.1" 200 None
2026-06-25 21:49:55,745 DEBUG aieng.syn_data.text.clients: *********** Extracted JSON payload: *********** 
{
  "question": "According to the Consumer Credit Card Agreement, what specific document is incorporated into and forms a part of the Agreement itself?",
  "gold_answer": "The Account Opening Disclosure (referred to as the \"Disclosure\") is incorporated into and is part of the Agreement."
}
*********** End of JSON payload ***********
2026-06-25 21:49:55,746 INFO aieng.syn_data.text.generation: Payload: {'question': 'According to the Consumer Credit Card Agreement, what specific document is incorporated into and forms a part of the Agreement itself?', 'gold_answer': 'The Account Opening Disclosure (referred to as the "Disclosure"

╭──────────────────────────────────────── Generation Strategy: zero_shot ─────────────────────────────────────────╮
│ zero_shot                                                                                                       │
│ Q: According to the Consumer Credit Card Agreement, what specific document is incorporated into and forms a     │
│ part of the Agreement itself?                                                                                   │
│ A: The Account Opening Disclosure (referred to as the "Disclosure") is incorporated into and is part of the     │
│ Agreement.                                                                                                      │
│ Failure mode: None                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────── Generation Strategy: one_shot ─────────────────────────────────────────╮
│ one_shot                                                                                                        │
│ Q: According to the agreement, what specific document is integrated into and forms a part of the Consumer       │
│ Credit Card Agreement?                                                                                          │
│ A: The Credit Card Account Opening Disclosure is incorporated into and is part of the Consumer Credit Card      │
│ Agreement.                                                                                                      │
│ Failure mode: None                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────── Generation Strategy: few_shot ─────────────────────────────────────────╮
│ few_shot                                                                                                        │
│ Q: According to the Consumer Credit Card Agreement, what specific document is incorporated into and forms a     │
│ part of the Agreement?                                                                                          │
│ A: The Credit Card Account Opening Disclosure is incorporated into and is part of the Agreement.                │
│ Failure mode: None                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────── Generation Strategy: topic_controlled ─────────────────────────────────────╮
│ topic_controlled                                                                                                │
│ Q: How does the Consumer Credit Card Agreement integrate the document referred to as the Credit Card Account    │
│ Opening Disclosure?                                                                                             │
│ A: The Credit Card Account Opening Disclosure is incorporated directly into the Consumer Credit Card Agreement  │
│ and is legally considered a part of the Agreement.                                                              │
│ Failure mode: None                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Quick pick guide

Start simple → zero-shot 

Want consistent format → one/few-shot 

Want variety from long paragraphs → topic-controlled 

## 3. Save raw synthetic samples

Store outputs from each strategy for quality filtering in Step 3.

In [5]:
raw_samples = generate_raw_synthetic_corpus(
    teacher,
    train_paragraphs,
    max_paragraphs=len(train_paragraphs),
)
console.print(f"Raw synthetic samples: {len(raw_samples)}")

save_typed_jsonl(
    SYNTHETIC_RAW_PATH,
    raw_samples,
    to_dict=lambda sample: sample.to_dict(),
)
SYNTHETIC_RAW_PATH

2026-06-25 21:54:29,877 DEBUG urllib3.connectionpool: Starting new HTTPS connection (1): proxy.vectorinstitute.ai:443
2026-06-25 21:54:31,806 DEBUG urllib3.connectionpool: https://proxy.vectorinstitute.ai:443 "POST /v1/chat/completions HTTP/1.1" 200 None
2026-06-25 21:54:31,808 DEBUG aieng.syn_data.text.clients: *********** Extracted JSON payload: *********** 
{
  "question": "According to the Consumer Credit Card Agreement, what specific document is incorporated into and forms a part of the Agreement itself?",
  "gold_answer": "The Account Opening Disclosure (referred to as the \"Disclosure\") is incorporated into and is part of the Agreement."
}
*********** End of JSON payload ***********
2026-06-25 21:54:31,809 INFO aieng.syn_data.text.generation: Payload: {'question': 'According to the Consumer Credit Card Agreement, what specific document is incorporated into and forms a part of the Agreement itself?', 'gold_answer': 'The Account Opening Disclosure (referred to as the "Disclosure"

Raw synthetic samples: 144

PosixPath('implementations/qa_text_generation/data/synthetic/synthetic_raw.jsonl')